<a href="https://colab.research.google.com/github/gee666thu/r5-machine-learning-lab/blob/main/modf%20exp4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

In [2]:
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target
print("Dataset Shape:", X.shape)
print("Class distribution:", np.bincount(y), "(0: Malignant, 1: Benign)")

Dataset Shape: (569, 30)
Class distribution: [212 357] (0: Malignant, 1: Benign)


In [3]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [4]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)
print("Preprocessing complete.")

Preprocessing complete.


In [5]:
mle = LogisticRegression(penalty=None, max_iter=10000)
mle.fit(X_train, y_train)

LogisticRegression(max_iter=10000, penalty=None)

In [6]:
l2 = LogisticRegression(penalty="l2", C=1.0, max_iter=10000)
l2.fit(X_train, y_train)

LogisticRegression(max_iter=10000)

In [7]:
l1 = LogisticRegression(penalty="l1", solver="liblinear", C=1.0, max_iter=10000)
l1.fit(X_train, y_train)

LogisticRegression(max_iter=10000, penalty='l1', solver='liblinear')

In [9]:
weights = pd.DataFrame()
weights["Feature"] = ["Intercept"] + list(data.feature_names)
weights["MLE"] = np.insert(mle.coef_[0], 0, mle.intercept_[0])
weights["MAP_L2 (Gaussian)"] = np.insert(l2.coef_[0], 0, l2.intercept_[0])
weights["MAP_L1 (Laplace)"] = np.insert(l1.coef_[0], 0, l1.intercept_[0])
print("\nParameter Estimates:")
print(weights.to_string(index=False))


Parameter Estimates:
                Feature         MLE  MAP_L2 (Gaussian)  MAP_L1 (Laplace)
              Intercept  -63.532600           0.445585          0.000000
            mean radius    9.433096          -0.431904          0.000000
           mean texture  -16.986388          -0.387326          0.000000
         mean perimeter   40.020926          -0.393432          0.000000
              mean area   10.890150          -0.465210          0.000000
        mean smoothness    5.033443          -0.071667          0.000000
       mean compactness  274.940098           0.540164          0.000000
         mean concavity -134.838158          -0.801458          0.000000
    mean concave points -266.184736          -1.119804         -2.416668
          mean symmetry   40.996505           0.236119          0.023455
 mean fractal dimension -168.017946           0.075921          0.000000
           radius error -259.639462          -1.268178         -2.485888
          texture error   35.

In [10]:
def evaluate(model, name):
    pred = model.predict(X_test)
    prob = model.predict_proba(X_test)[:, 1]
    acc = accuracy_score(y_test, pred)
    pre = precision_score(y_test, pred, zero_division=0)
    rec = recall_score(y_test, pred, zero_division=0)
    f1 = f1_score(y_test, pred, zero_division=0)
    auc = roc_auc_score(y_test, prob)
    return {"Model": name, "Accuracy": round(acc, 2), "Precision": round(pre, 2), "Recall": round(rec, 2), "F1-Score": round(f1, 2), "AUC-ROC": round(auc, 2)}

In [11]:
r1 = evaluate(mle, "MLE")
r2 = evaluate(l2, "MAP L2")
r3 = evaluate(l1, "MAP L1")
df = pd.DataFrame([r1, r2, r3])
print("\nPerformance Comparison:")
print(df.to_string(index=False))


Performance Comparison:
 Model  Accuracy  Precision  Recall  F1-Score  AUC-ROC
   MLE      0.94       0.98    0.92      0.95     0.98
MAP L2      0.97       0.97    0.99      0.98     1.00
MAP L1      0.97       0.99    0.97      0.98     1.00


In [12]:
print("\nNumber of zero weights in L1 model:", np.sum(l1.coef_[0] == 0))
print("Total number of features:", len(data.feature_names))


Number of zero weights in L1 model: 15
Total number of features: 30


In [13]:
print("\nResult: MLE and MAP Logistic Regression models were compared successfully.")


Result: MLE and MAP Logistic Regression models were compared successfully.
